# RiverWatch2 — exact-Kratzert day-1 eval (Kaggle GPU, full-531)

Runs `scripts/eval_day1_kratzert.py` (rolling lead-1, one prediction per calendar day) for the full-531 δHBV headline — the number directly comparable to the 0.83 record. LSTM members are cheap; δHBV needs the GPU. Validate against the ~0.808 LSTM-ensemble band first.


In [ ]:
# --- Setup: clone repo (SHA-logged), install the few deps Kaggle lacks ---
import os, subprocess, sys, textwrap
os.chdir('/kaggle/working')
if not os.path.exists('riverwatch2'):
    subprocess.run(['git','clone','--depth','1','--branch','benchmark-competition-2026-07',
                    'https://github.com/andrewnakas/riverwatch2.git'], check=True)
os.chdir('/kaggle/working/riverwatch2')
sha = subprocess.run(['git','rev-parse','--short','HEAD'],
                     capture_output=True, text=True).stdout.strip()
print('repo SHA:', sha)
# Kaggle already has torch/pandas/numpy/scikit-learn; nothing else is needed.
os.environ['RW2_ENABLE_MBLSTM'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'


In [ ]:
# --- Wire Kaggle Dataset inputs to repo-relative paths ---
import glob, shutil, os
# Static attrs + gauge ids + station registry (load-bearing: without
# camels_attrs.json the static overlay is all-NaN and NSE craters to 0.40).
STATIC_DS = '/kaggle/input/rw2-camels-static'
for f in ['camels_attrs.json','camels_gauge_ids.json','stations_40_enriched.json']:
    src = os.path.join(STATIC_DS, f)
    if os.path.exists(src):
        shutil.copy(src, f'data/{f}')
        print('staged', f)
    else:
        print('WARNING missing static input:', src)
# Corpora: one Dataset per forcing. Resolve the dir that holds the 531 csv.gz.
def corpus_dir(forcing):
    cands = glob.glob(f'/kaggle/input/rw2-camels-corpus-{forcing}/**/camels_corpus_{forcing}_v2',
                      recursive=True) or \
            glob.glob(f'/kaggle/input/rw2-camels-corpus-{forcing}/**/*.csv.gz', recursive=True)
    if not cands: raise FileNotFoundError(f'no corpus for {forcing}')
    d = cands[0]
    return d if os.path.isdir(d) else os.path.dirname(d)
for F in ['daymet','maurer','nldas']:
    try: print(F, '->', corpus_dir(F), len(glob.glob(corpus_dir(F)+'/*.csv.gz')), 'basins')
    except Exception as e: print(F, 'NOT MOUNTED', e)


In [ ]:
import subprocess, glob
FORCING = 'daymet'
CORPUS = corpus_dir(FORCING)
cks = sorted(glob.glob(f'/kaggle/input/rw2-noq-ckpts/**/camels531_{FORCING}_*combined100_s*.pt',
                       recursive=True)) or \
      sorted(glob.glob(f'/kaggle/input/**/camels531_{FORCING}_v2r_s*.pt', recursive=True))
ckpt = ':'.join(cks[:3])
cmd = (f'python scripts/eval_day1_kratzert.py --ckpt {ckpt} --corpus-dir {CORPUS} '
       f'--start 1989-10-01 --end 1999-09-30 --stride-days 1 --camels-subset 531 '
       f'--label {FORCING}_day1_full531')
print('>>', cmd, flush=True)
subprocess.run(cmd, shell=True)
